# Random Forest

## 1. Load Data

In [ ]:
import sys
sys.path.append('../src')
from data_prep import load_data

X_train, X_test, y_train, y_test, amount_test = load_data()

## 2. Establish Baseline

Default parameters, kept as a reference point for the tuning step below.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
import matplotlib.pyplot as plt


# 1. Initialize the Random Forest with 100 trees
# random_state=42 ensures you get the same result every time you run it
rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42)

# 2. Train it on the full feature set
rf_baseline.fit(X_train, y_train)

# 3. Predict the results for our test set
y_pred_baseline = rf_baseline.predict(X_test)

# 4. Print the Accuracy Score
rf_acc = accuracy_score(y_test, y_pred_baseline)
print(f"Random Forest Accuracy: {rf_acc*100:.2f}%")

# 5. Create the Confusion Matrix to check those False Negatives
cm_rf = confusion_matrix(y_test, y_pred_baseline)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=['Legitimate', 'Fraudulent'])
disp_rf.plot(cmap='Purples')
plt.title("Random Forest Baseline: Fraud Detection Confusion Matrix")
plt.show()

## 3. Hyperparameter Tuning

Two independent `RandomizedSearchCV` attempts (different search spaces) were run against 3-fold cross-validated PR-AUC. Both lost to the plain default `RandomForestClassifier(n_estimators=100)` (baseline CV PR-AUC ≈0.841 vs. best tuned ≈0.839–0.840), and each search took 30+ minutes on this dataset since sklearn's RF lacks a fast histogram-based split-finder like XGBoost's. Given two independent losses, further tuning isn't worth the cost here — the baseline is kept as the final model.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

baseline_cv_score = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    X_train, y_train, scoring='average_precision',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
).mean()

print(f"Baseline CV PR-AUC: {baseline_cv_score:.4f}")
print("Using the baseline as the final model (tuning did not improve on it -- see note above).")

rf_model = rf_baseline

## 4. PR-AUC & Threshold Tuning

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score, classification_report
import numpy as np
import matplotlib.pyplot as plt

rf_probs = rf_model.predict_proba(X_test)[:, 1]
y_pred_rf = rf_model.predict(X_test)

# --- Default threshold (0.5), for reference ---
print("Classification Report (default threshold = 0.5, tuned model):")
print(classification_report(y_test, y_pred_rf))

# --- PR-AUC: a better summary metric than ROC-AUC when classes are this imbalanced ---
pr_auc = average_precision_score(y_test, rf_probs)
print(f"PR-AUC (Average Precision): {pr_auc:.4f}")

precision, recall, thresholds = precision_recall_curve(y_test, rf_probs)

# --- Find the threshold that maximizes F1, instead of picking an arbitrary recall floor ---
f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
best_f1_idx = np.argmax(f1_scores)
best_f1_threshold = thresholds[best_f1_idx]

print(f"\nF1-maximizing threshold: {best_f1_threshold:.4f}")
print(f"Precision: {precision[best_f1_idx]:.4f}  Recall: {recall[best_f1_idx]:.4f}  F1: {f1_scores[best_f1_idx]:.4f}")

y_pred_f1 = (rf_probs >= best_f1_threshold).astype(int)
print("\nConfusion Matrix (F1-optimal threshold):")
print(confusion_matrix(y_test, y_pred_f1))
print("\nClassification Report (F1-optimal threshold):")
print(classification_report(y_test, y_pred_f1))

# --- Visualize the full precision-recall trade-off ---
plt.figure(figsize=(7, 5))
plt.plot(recall, precision, label="PR curve")
plt.scatter(
    recall[best_f1_idx], precision[best_f1_idx],
    color="red", zorder=5, label=f"F1-optimal (t={best_f1_threshold:.3f})"
)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Random Forest: Precision-Recall Curve")
plt.legend()
plt.show()

## 5. What if you specifically need higher recall?

Not a recommendation — just illustrating the precision cost of forcing a specific recall floor, e.g. if missed fraud has a known dollar cost that makes 90% recall a hard requirement.

In [ ]:
target_recall = 0.90
idxs = np.where(recall[:-1] >= target_recall)[0]  # drop last point (recall=0 has no threshold)

if len(idxs) > 0:
    best_idx = idxs[np.argmax(precision[idxs])]
    chosen_threshold = thresholds[best_idx]
    print(f"Chosen threshold: {chosen_threshold:.4f}")
    print(f"Precision at threshold: {precision[best_idx]:.4f}")
    print(f"Recall at threshold: {recall[best_idx]:.4f}")

    y_pred_tuned = (rf_probs >= chosen_threshold).astype(int)
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred_tuned))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_tuned))
else:
    print("Model cannot reach that recall with meaningful thresholds.")

## 6. Cost-Based Threshold

Instead of a generic F1 balance, tie the threshold to real dollars: a missed fraud (false negative) costs its transaction `Amount`; a false alarm (false positive) costs a fixed review cost. Pick the threshold that minimizes total expected cost.

In [ ]:
REVIEW_COST = 10  # assumed dollar cost to investigate one flagged transaction -- adjust to your own estimate

y_test_arr = y_test.values
amount_arr = amount_test.values

costs = []
for t in thresholds:
    y_pred_t = (rf_probs >= t).astype(int)
    fn_mask = (y_pred_t == 0) & (y_test_arr == 1)
    fp_mask = (y_pred_t == 1) & (y_test_arr == 0)
    cost = amount_arr[fn_mask].sum() + REVIEW_COST * fp_mask.sum()
    costs.append(cost)

costs = np.array(costs)
best_cost_idx = np.argmin(costs)
best_cost_threshold = thresholds[best_cost_idx]

print(f"Cost-minimizing threshold: {best_cost_threshold:.4f}")
print(f"Total cost at this threshold: ${costs[best_cost_idx]:,.2f}")
print(f"Precision: {precision[best_cost_idx]:.4f}  Recall: {recall[best_cost_idx]:.4f}")

y_pred_cost = (rf_probs >= best_cost_threshold).astype(int)
print("\nConfusion Matrix (cost-optimal threshold):")
print(confusion_matrix(y_test, y_pred_cost))
print("\nClassification Report (cost-optimal threshold):")
print(classification_report(y_test, y_pred_cost))

plt.figure(figsize=(7, 5))
plt.plot(thresholds, costs)
plt.scatter(best_cost_threshold, costs[best_cost_idx], color="red", zorder=5, label=f"Min cost (t={best_cost_threshold:.3f})")
plt.xlabel("Threshold")
plt.ylabel("Total cost ($)")
plt.title("Random Forest: Cost vs. Threshold")
plt.legend()
plt.show()

## 7. Feature Importance

Before we guess which features matter, ask the model what it thinks is important.

In [ ]:
import pandas as pd

# Extract feature importances
importances = rf_model.feature_importances_

# Use the actual column names from X_train
feature_names = X_train.columns

# Create DataFrame
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(feature_importance_df)

## 8. Save Model & Predictions

So `06_model_comparison.ipynb` can load results without retraining.

In [ ]:
import json
import joblib

joblib.dump(rf_model, '../models/rf.pkl')
np.save('../predictions/rf_probs.npy', rf_probs)

with open('../models/rf_thresholds.json', 'w') as f:
    json.dump({
        "f1_optimal_threshold": float(best_f1_threshold),
        "cost_optimal_threshold": float(best_cost_threshold),
        "review_cost_assumption": REVIEW_COST,
        "model_selected": "baseline",
        "baseline_cv_pr_auc": float(baseline_cv_score),
        "tuning_note": "Two RandomizedSearchCV attempts (different search spaces) both scored lower than the baseline under 3-fold CV PR-AUC (~0.839-0.840 vs ~0.841); tuning was dropped in favor of the baseline.",
    }, f, indent=2)